In [ ]:
import requests
import pandas as pd
import torchvision
import torch

In [ ]:
with open('downloader.py', 'wb') as f:
    f.write(requests.get('https://raw.githubusercontent.com/openimages/dataset/master/downloader.py').content)

In [ ]:
classes = pd.read_csv(
    'https://storage.googleapis.com/openimages/2018_04/class-descriptions-boxable.csv',
    header=None, names=["LabelID", "LabelName"]
)
target_classes = ["Plate", "Spoon", "Book"]

# Get LabelIDs (used in annotations) for those class names
target_labels = classes[classes["LabelName"].isin(target_classes)]["LabelID"].tolist()

# Load annotations
annotations = pd.read_csv(
    'https://storage.googleapis.com/openimages/2018_04/train/train-annotations-bbox.csv'
)

# Save Locally
annotations.to_csv('train-annotations-bbox.csv', index=False)
classes.to_csv('class-descriptions-boxable.csv', index=False)

# Filter annotations by selected target labels
filtered = annotations[annotations['LabelName'].isin(target_labels)]

# Get unique image IDs for those annotations
image_ids = filtered['ImageID'].unique()

# Save the image IDs with the "train/" prefix for downloading later
with open('my_image_list.txt', 'w') as f:
    for img_id in image_ids:
        f.write(f"train/{img_id}\n")

In [ ]:
!python downloader.py my_image_list.txt --download_folder=custom_images --num_processes=5

In [ ]:
coco_labels = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign',
    'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack', 'umbrella',
    'N/A', 'N/A', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard',
    'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard',
    'surfboard', 'tennis racket', 'bottle', 'N/A', 'wine glass', 'cup', 'fork',
    'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange',
    'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch',
    'potted plant', 'bed', 'N/A', 'dining table', 'N/A', 'N/A', 'toilet',
    'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone',
    'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A', 'book',
    'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

In [ ]:
"""
Function to filter at least a sample of 100 images for training the model
"""
target_classes = ["Plate", "Spoon", "Book"]
classes = pd.read_csv("class-descriptions-boxable.csv", header=None)
target_label_names = classes[classes[1].isin(target_classes)][0].tolist()

# Load full annotations
full_annots = pd.read_csv("train-annotations-bbox.csv")

# Filter annotations for only those target classes
filtered_annots = full_annots[full_annots['LabelName'].isin(target_label_names)]

# Select the first 100 unique images
selected_image_ids = filtered_annots['ImageID'].unique()[:100]

subset_annots = filtered_annots[filtered_annots['ImageID'].isin(selected_image_ids)]

selected_image_ids = filtered_annots['ImageID'].unique()[:100]
subset_annots = filtered_annots[filtered_annots['ImageID'].isin(selected_image_ids)]

subset_annots.to_csv("subset-train-annotations-bbox.csv", index=False)

import os
import shutil

source_folder = "custom_images"
target_folder = "custom_images_100"
os.makedirs(target_folder, exist_ok=True)

for img_id in selected_image_ids:
    src_path = os.path.join(source_folder, f"{img_id}.jpg")
    dst_path = os.path.join(target_folder, f"{img_id}.jpg")
    if os.path.exists(src_path):
        shutil.copy(src_path, dst_path)

In [ ]:
import torch
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from PIL import Image

'''
This class definition to Load the Open Images Dataset.
'''
from torchvision.transforms import functional as F

class OpenImagesDataset(Dataset):
    def __init__(self, root, annotation_csv, classes_csv, target_classes, transforms=None):
        self.root = root
        self.transforms = transforms
        self.annots = pd.read_csv(annotation_csv)
        self.classes = pd.read_csv(classes_csv, header=None)

        # Get target label codes
        self.target_classes = target_classes
        self.label_names = self.classes[self.classes[1].isin(target_classes)][0].tolist()

        # Filter only relevant annotations
        self.annots = self.annots[self.annots['LabelName'].isin(self.label_names)]
        self.image_ids = self.annots['ImageID'].unique()

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        img_path = os.path.join(self.root, image_id + '.jpg')
        img = Image.open(img_path).convert("RGB")

        boxes = []
        labels = []
        image_annots = self.annots[self.annots['ImageID'] == image_id]
        for _, row in image_annots.iterrows():
            xmin = row['XMin'] * img.width
            xmax = row['XMax'] * img.width
            ymin = row['YMin'] * img.height
            ymax = row['YMax'] * img.height
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(self.label_names.index(row['LabelName']) + 1)

        if len(boxes) == 0:
            return self.__getitem__((idx + 1) % len(self))  # skip image with no boxes

        boxes = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        target = {"boxes": boxes, "labels": labels, "image_id": torch.tensor([idx])}

        if self.transforms:
            img = self.transforms(img)

        return img, target

    def __len__(self):
        return len(self.image_ids)

# Transformations
transform = T.Compose([T.ToTensor()])

# Target classes
target_classes = ["Plate", "Spoon", "Book"]

# Dataset and DataLoader
dataset = OpenImagesDataset(
    root="custom_images_100",
    annotation_csv="subset-train-annotations-bbox.csv",
    classes_csv="class-descriptions-boxable.csv",
    target_classes=target_classes,
    transforms=transform
)

def collate_fn(batch):
    return tuple(zip(*batch))

data_loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

# Load pre-trained Faster R-CNN model
model = fasterrcnn_resnet50_fpn(pretrained=True)

# Replace the box predictor
num_classes = len(target_classes) + 1
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

# Move model to device
device = torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')
model.to(device)

In [ ]:
from tqdm import tqdm, trange
import torch.optim as optim
import os

params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

num_epochs = 3

for epoch in trange(num_epochs, desc="Epochs"):
    model.train()
    running_loss = 0.0

    for images, targets in tqdm(data_loader, desc=f"Training Epoch {epoch+1}", leave=False):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        running_loss += losses.item()

    avg_loss = running_loss / len(data_loader)
    print(f"Epoch {epoch+1} completed. Avg Loss: {avg_loss:.4f}")
    # Save the model
    torch.save(model.state_dict(), f"fasterrcnn_epoch_{epoch+1}.pth")